# 13.3: Time to first token vs. time per output token

Which metric matters depends on the application. For a chat interface that streams tokens as they are generated (Section 8), TTFT determines how responsive the system feels, and TPOT only needs to be faster than a person reads. For a coding agent that must finish a long generation before acting, end-to-end latency matters most, and TPOT dominates it. For summarizing a long document into a short answer, prefill (and thus TTFT) dominates; for writing a long essay from a short prompt, decode dominates.

A small calculator makes the tradeoffs concrete:


In [ ]:
def latency_estimate(n_params, bytes_per_param, prompt_len, gen_len,
                     peak_flops=1e15, bandwidth=3e12, mfu=0.5, mbu=0.7):
    """Rough single-request latency using compute and memory lower bounds.
    mfu/mbu: fraction of peak FLOP/s and peak bandwidth actually achieved (assumed)."""
    weight_bytes = n_params * bytes_per_param
    prefill_compute = 2 * n_params * prompt_len / (peak_flops * mfu)
    prefill_memory = weight_bytes / (bandwidth * mbu)
    ttft = max(prefill_compute, prefill_memory)
    tpot = weight_bytes / (bandwidth * mbu)      # ignores KV-cache reads at short context
    return ttft, tpot, ttft + (gen_len - 1) * tpot

ttft, tpot, total = latency_estimate(8e9, 2, prompt_len=2000, gen_len=500)
print(f"TTFT {ttft*1e3:.0f} ms, TPOT {tpot*1e3:.1f} ms, total {total:.2f} s")
# TTFT 64 ms, TPOT 7.6 ms, total 3.87 s
